In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
df = pd.read_excel('./data/cleaned_dataset.xlsx')
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)

f:\app space\Anaconda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Shape: (60, 17)

Columns: ['customer_id', 'first_name', 'gender', 'age', 'city', 'province', 'signup_date', 'membership_tier', 'purchase_count', 'avg_order_value', 'total_spending', 'last_purchase_days', 'payment_method', 'device', 'discount_used', 'returned_items', 'satisfaction_score']

Dtypes:
customer_id                    int64
first_name                       str
gender                           str
age                            int64
city                             str
province                         str
signup_date           datetime64[us]
membership_tier                  str
purchase_count                 int64
avg_order_value              float64
total_spending               float64
last_purchase_days             int64
payment_method                   str
device                           str
discount_used                    str
returned_items                 int64
satisfaction_score             int64
dtype: object


check the dataset health

In [2]:
required_columns = [
"customer_id", "first_name", "gender", "age", "city", "province",
"signup_date", "membership_tier", "purchase_count", "avg_order_value",
"total_spending", "last_purchase_days", "payment_method", "device",
"discount_used", "returned_items", "satisfaction_score"
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"ندارند وجود فایل در ستونها این: {missing_columns}")
df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")
numeric_columns = [
"age", "purchase_count", "avg_order_value", "total_spending",
"last_purchase_days", "returned_items", "satisfaction_score"
]
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
text_columns = [
"first_name", "gender", "city", "province", "membership_tier",
"payment_method", "device", "discount_used"
]
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
print("دیتاست ابعاد:", df.shape)
print("تکراری رکوردهای تعداد:", df.duplicated().sum())
print("ستون هر گمشده مقادیر:")
print(df.isna().sum().sort_values(ascending=False))

دیتاست ابعاد: (60, 17)
تکراری رکوردهای تعداد: 0
ستون هر گمشده مقادیر:
customer_id           0
first_name            0
gender                0
age                   0
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        0
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64


In [3]:
df.head(2)

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,M,19,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3


In [4]:
import plotly.express as px
import pandas as pd

# (make sure analysis_df already exists with age_group, membership_tier, etc.)
analysis_df = df.dropna(subset=["gender", "age", "membership_tier", "device", "payment_method"]).copy()

age_bins = [0, 24, 34, 44, 54, 64, np.inf]
age_labels = ["25 زیر", "34-25", "44-35", "54-45", "64-55", "بیشتر و 65"]
analysis_df["age_group"] = pd.cut(
    analysis_df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

# Profile summary
profile_summary = {
    "تعداد مشتریان": analysis_df["customer_id"].nunique(),
    "میانگین سن": analysis_df["age"].mean(),
    "میانه سن": analysis_df["age"].median(),
    "رایج‌ترین جنسیت": analysis_df["gender"].mode().iloc[0],
    "رایج‌ترین سطح عضویت": analysis_df["membership_tier"].mode().iloc[0],
    "رایج‌ترین دستگاه": analysis_df["device"].mode().iloc[0],
    "رایج‌ترین روش پرداخت": analysis_df["payment_method"].mode().iloc[0]
}
profile_table = pd.Series(profile_summary, name="مقدار")
print(profile_table)
# 1. Age group distribution
fig_age = px.histogram(
    analysis_df,
    x="age_group",
    title="<b>تعداد مشتریان در گروه‌های سنی</b>",
    color_discrete_sequence=["#636EFA"],      # same colour as your original plot
)
fig_age.update_layout(
    width=900,
    height=500,
    bargap=0.1,
    xaxis_title="گروه سنی",
    yaxis_title="تعداد مشتری"
)
fig_age.show()

# 2. Membership tier distribution (horizontal bars)
fig_membership = px.histogram(
    analysis_df,
    y="membership_tier",                      # horizontal bar
    title="<b>توزیع سطح عضویت مشتریان</b>",
    color_discrete_sequence=["#EF553B"],
)
fig_membership.update_layout(
    width=800,
    height=450,
    bargap=0.1,
    xaxis_title="تعداد مشتری",
    yaxis_title=""
)
fig_membership.show()

# 3. Device distribution (horizontal)
fig_device = px.histogram(
    analysis_df,
    y="device",
    title="<b>دستگاه مورد استفاده مشتریان</b>",
    color_discrete_sequence=["#00CC96"],
)
fig_device.update_layout(
    width=800,
    height=450,
    bargap=0.1,
    xaxis_title="تعداد مشتری",
    yaxis_title=""
)
fig_device.show()

# 4. Payment method distribution (horizontal)
fig_payment = px.histogram(
    analysis_df,
    y="payment_method",
    title="<b>روش پرداخت مشتریان</b>",
    color_discrete_sequence=["#AB63FA"],
)
fig_payment.update_layout(
    width=800,
    height=450,
    bargap=0.1,
    xaxis_title="تعداد مشتری",
    yaxis_title=""
)
fig_payment.show()

تعداد مشتریان                      60
میانگین سن                  45.366667
میانه سن                         45.0
رایج‌ترین جنسیت                     M
رایج‌ترین سطح عضویت              Gold
رایج‌ترین دستگاه              Android
رایج‌ترین روش پرداخت    Online Wallet
Name: مقدار, dtype: object


In [5]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Helper: build each bar trace
def bar_trace(data, col, orientation, color):
    counts = data[col].value_counts().reset_index()
    counts.columns = ['category', 'count']
    if orientation == 'h':
        return go.Bar(y=counts['category'], x=counts['count'],
                      marker_color=color, orientation='h', name=col)
    else:
        return go.Bar(x=counts['category'], y=counts['count'],
                      marker_color=color, name=col)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("تعداد مشتریان در هر گروه سنی",
                                    "توزیع سطح عضویت",
                                    "دستگاه مورد استفاده",
                                    "روش پرداخت"),
                    vertical_spacing=0.12, horizontal_spacing=0.1)

# Age (vertical bar)
fig.add_trace(bar_trace(analysis_df, "age_group", "v", "#636EFA"), row=1, col=1)
fig.update_xaxes(title_text="گروه سنی", row=1, col=1)
fig.update_yaxes(title_text="تعداد مشتری", row=1, col=1)

# Membership (horizontal)
fig.add_trace(bar_trace(analysis_df, "membership_tier", "h", "#EF553B"), row=1, col=2)
fig.update_xaxes(title_text="تعداد مشتری", row=1, col=2)

# Device (horizontal)
fig.add_trace(bar_trace(analysis_df, "device", "h", "#00CC96"), row=2, col=1)
fig.update_xaxes(title_text="تعداد مشتری", row=2, col=1)

# Payment (horizontal)
fig.add_trace(bar_trace(analysis_df, "payment_method", "h", "#AB63FA"), row=2, col=2)
fig.update_xaxes(title_text="تعداد مشتری", row=2, col=2)

fig.update_layout(
    width=1200,
    height=800,
    showlegend=False,
    title_text="<b>داشبورد پروفایل مشتریان</b>",
    title_x=0.5,
    bargap=0.2
)
fig.show()

با توجه به نمودار : سطح عضویت توی قسمت silver ضعیف عمل کرده است 
و توی قسمت گروه سنی میتونیم روی  مشتری های زیر 25 سرمایه گذاری بکنیم

In [6]:
df.head()

,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,M,19,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3
2,1003,Parsa,M,31,Shiraz,Fars,2023-06-20,Gold,21,59.46,1248.66,22,Online Wallet,iPhone,Yes,6,1
3,1004,Sina,M,58,Mashhad,Khorasan,2021-11-08,Gold,23,266.15,6121.45,40,Card,Android,No,4,4
4,1005,Kimia,M,28,Isfahan,Isfahan,2021-10-21,Silver,23,169.54,3899.42,273,Online Wallet,Android,Yes,7,4


In [7]:
import plotly.express as px

# Count genders
gender_counts = analysis_df['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']

fig_gender = px.bar(
    gender_counts,
    x='gender',
    y='count',
    title="<b>توزیع جنسیت مشتریان</b>",          # Customer gender distribution
    color='gender',
    color_discrete_map={'M': "#63B8FA", 'F': "#EF3BBF"},   
    text='count'                               
)

fig_gender.update_layout(
    width=600,
    height=450,
    bargap=0.3,
    xaxis_title="جنسیت",
    yaxis_title="تعداد مشتری",
    showlegend=False
)

fig_gender.show()

باتوجه به این نمودار تعداد مشتری های خانم خیلی کمتر هستش نسبت به آقا

In [8]:
import plotly.express as px
import pandas as pd

# 1. Age
fig_age = px.histogram(analysis_df, x="age", nbins=15,
                       title="<b>Age Distribution</b>",
                       color_discrete_sequence=["#636EFA"])
fig_age.update_layout(width=700, height=400, bargap=0.1,
                      xaxis_title="Age", yaxis_title="Count")
fig_age.show()

# 2. Purchase count
fig_purch = px.histogram(analysis_df, x="purchase_count", nbins=20,
                         title="<b>Purchase Count Distribution</b>",
                         color_discrete_sequence=["#EF553B"])
fig_purch.update_layout(width=700, height=400, bargap=0.1,
                        xaxis_title="Purchase Count", yaxis_title="Count")
fig_purch.show()

# 3. Avg order value
fig_avg = px.histogram(analysis_df, x="avg_order_value", nbins=15,
                       title="<b>Average Order Value</b>",
                       color_discrete_sequence=["#00CC96"])
fig_avg.update_layout(width=700, height=400, bargap=0.1,
                      xaxis_title="Avg Order Value", yaxis_title="Count")
fig_avg.show()

# 4. Total spending
fig_total = px.histogram(analysis_df, x="total_spending", nbins=15,
                         title="<b>Total Spending</b>",
                         color_discrete_sequence=["#AB63FA"])
fig_total.update_layout(width=700, height=400, bargap=0.1,
                        xaxis_title="Total Spending", yaxis_title="Count")
fig_total.show()

# 5. Last purchase days
fig_last = px.histogram(analysis_df, x="last_purchase_days", nbins=15,
                        title="<b>Days Since Last Purchase</b>",
                        color_discrete_sequence=["#FFA15A"])
fig_last.update_layout(width=700, height=400, bargap=0.1,
                       xaxis_title="Days", yaxis_title="Count")
fig_last.show()

# 6. Returned items
fig_return = px.histogram(analysis_df, x="returned_items", nbins=10,
                          title="<b>Returned Items Distribution</b>",
                          color_discrete_sequence=["#19D3F3"])
fig_return.update_layout(width=700, height=400, bargap=0.1,
                         xaxis_title="Returned Items", yaxis_title="Count")
fig_return.show()

# 7. Satisfaction score
fig_sat = px.histogram(analysis_df, x="satisfaction_score", nbins=10,
                       title="<b>Satisfaction Score</b>",
                       color_discrete_sequence=["#FF6692"])
fig_sat.update_layout(width=700, height=400, bargap=0.1,
                      xaxis_title="Score", yaxis_title="Count")
fig_sat.show()

# ----- Categorical: bar chart (vertical unless too many categories) -----
# 8. City (top 10 by count to avoid clutter)
top_cities = analysis_df['city'].value_counts().nlargest(10).index
city_df = analysis_df[analysis_df['city'].isin(top_cities)]
fig_city = px.histogram(city_df, x="city", 
                        title="<b>Top 10 Cities</b>",
                        color_discrete_sequence=["#636EFA"])
fig_city.update_layout(width=900, height=450, bargap=0.3,
                       xaxis_title="City", yaxis_title="Count")
fig_city.show()



# ----- Time series: signup_date -----
# Aggregate signups by month and plot a line
signup_counts = analysis_df.set_index('signup_date').resample('ME').size().reset_index(name='count')
fig_signup = px.line(signup_counts, x='signup_date', y='count', markers=True,
                     title="<b>Monthly Signups</b>",
                     color_discrete_sequence=["#000000"])
fig_signup.update_layout(width=800, height=450,
                         xaxis_title="Month", yaxis_title="Number of Signups")
fig_signup.show()

1-بررسی سن 140
2-بررسی مقدار پرت توی total spending 

In [9]:
import plotly.express as px

fig = px.histogram(
    analysis_df,
    x="membership_tier",
    color="gender",
    barmode="group",                          # bars side‑by‑side
    title="<b>Membership Tier by Gender</b>",
    color_discrete_map={"M": "#63B1FA", "F": "#EF3BD4"},
    category_orders={"membership_tier": ["Bronze", "Silver", "Gold", "Platinum"]}  # order if needed
)
fig.update_layout(
    width=750,
    height=450,
    bargap=0.15,
    bargroupgap=0.1,
    xaxis_title="Membership Tier",
    yaxis_title="Count",
    legend_title="Gender"
)
fig.show()

نبود مشتری با platinum
و نسبت مشتری های مرد به زن در gold 

In [10]:
fig = px.histogram(
    analysis_df,
    x="device",
    color="membership_tier",
    barmode="group",
    title="<b>Device Used by Membership Tier</b>",
    color_discrete_sequence=px.colors.qualitative.Plotly   # automatic distinct colors
)
fig.update_layout(width=700, height=400, bargap=0.2,
                  xaxis_title="Device", yaxis_title="Count",
                  legend_title="Tier")
fig.show()

کسانی که ایفون داشته اند از هر مقدار پلن عضویت بجز silver به یک مقدار هستند کسانی که از webاستفاده کردهند از همه کمتر به vip خریدتد 

In [11]:
fig = px.histogram(
    analysis_df,
    x="membership_tier",
    y="total_spending",
    color="gender",
    barmode="group",
    histfunc="avg",                        # aggregates by average
    title="<b>Average Total Spending by Membership and Gender</b>",
    color_discrete_map={"M": "#2391F8", "F": "#EF3BE0"}
)
fig.update_layout(width=750, height=450, bargap=0.2,
                  xaxis_title="Membership Tier", yaxis_title="Avg Total Spending",
                  legend_title="Gender")
fig.show()

In [12]:
# Keep only top 10 cities to avoid clutter
top_cities = analysis_df['city'].value_counts().nlargest(10).index
city_gender = analysis_df[analysis_df['city'].isin(top_cities)]

fig1 = px.histogram(city_gender, x='city', color='gender',
                    barmode='group',
                    title='<b>City Distribution by Gender (Top 10)</b>',
                    color_discrete_map={'M': '#636EFA', 'F': '#EF553B'})
fig1.update_layout(width=900, height=450, bargap=0.3,
                   xaxis_title='City', yaxis_title='Count',
                   legend_title='Gender')
fig1.show()

In [13]:
fig2 = px.histogram(analysis_df, x='membership_tier', color='gender',
                    barmode='group',
                    title='<b>Membership Tier by Gender</b>',
                    color_discrete_map={'M': '#636EFA', 'F': '#EF553B'},
                    category_orders={'membership_tier': ['Bronze', 'Silver', 'Gold', 'Platinum']})
fig2.update_layout(width=700, height=450, bargap=0.2,
                   xaxis_title='Membership Tier', yaxis_title='Count',
                   legend_title='Gender')
fig2.show()

In [14]:
avg_sat = analysis_df.groupby('gender')['satisfaction_score'].mean().reset_index()
fig3a = px.bar(avg_sat, x='gender', y='satisfaction_score',
               color='gender',
               title='<b>Average Satisfaction Score by Gender</b>',
               color_discrete_map={'M': '#636EFA', 'F': '#EF553B'},
               text='satisfaction_score')
fig3a.update_layout(width=500, height=400, bargap=0.3,
                    xaxis_title='Gender', yaxis_title='Avg Satisfaction Score',
                    showlegend=False)
fig3a.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig3a.show()

In [15]:
fig3b = px.histogram(analysis_df, x='satisfaction_score', color='gender',
                     barmode='overlay',           # semi‑transparent overlapping
                     nbins=10,
                     title='<b>Satisfaction Score Distribution by Gender</b>',
                     color_discrete_map={'M': '#636EFA', 'F': '#EF553B'})
fig3b.update_layout(width=700, height=450, bargap=0.1,
                    xaxis_title='Satisfaction Score', yaxis_title='Count',
                    legend_title='Gender')
fig3b.show()

In [16]:
top_cities = analysis_df['city'].value_counts().nlargest(10).index
city_avg_sat = (analysis_df[analysis_df['city'].isin(top_cities)]
                .groupby('city')['satisfaction_score']
                .mean().reset_index()
                .sort_values('satisfaction_score', ascending=True))

fig4 = px.bar(city_avg_sat, y='city', x='satisfaction_score',
              orientation='h',
              title='<b>Average Satisfaction Score by City (Top 10)</b>',
              color='satisfaction_score',
              color_continuous_scale='blues')
fig4.update_layout(width=700, height=500,
                   xaxis_title='Avg Satisfaction Score', yaxis_title='City')
fig4.show()

In [17]:
top_cities = analysis_df['city'].value_counts().nlargest(8).index   # keep fewer to avoid crowding
city_tier = analysis_df[analysis_df['city'].isin(top_cities)]

fig5 = px.histogram(city_tier, x='city', color='membership_tier',
                    barmode='group',
                    title='<b>Membership Tier Distribution by City</b>',
                    category_orders={'membership_tier': ['Bronze', 'Silver', 'Gold', 'Platinum']})
fig5.update_layout(width=950, height=450, bargap=0.25,
                   xaxis_title='City', yaxis_title='Count',
                   legend_title='Tier')
fig5.show()

RFM analysis

In [18]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [19]:
rfm = df[["customer_id", "first_name", "total_spending", "purchase_count",
          "last_purchase_days", "membership_tier", "satisfaction_score",
          "age", "gender", "city", "province"]].copy()

# Rename for clarity
rfm.rename(columns={
    "total_spending": "Monetary",
    "purchase_count": "Frequency",
    "last_purchase_days": "Recency"
}, inplace=True)

# Recency: lower is better (more recent)
# Frequency: higher is better
# Monetary: higher is better

# 3. RFM Scoring using quintiles (1-5)
# Recency: inverted (1 = most recent, 5 = least recent)
rfm["R_Score"] = pd.qcut(rfm["Recency"], q=5, labels=[5, 4, 3, 2, 1], duplicates="drop")
# Frequency: higher = better
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), q=5, labels=[1, 2, 3, 4, 5], duplicates="drop")
# Monetary: higher = better
rfm["M_Score"] = pd.qcut(rfm["Monetary"], q=5, labels=[1, 2, 3, 4, 5], duplicates="drop")

# Convert to numeric
rfm["R_Score"] = rfm["R_Score"].astype(int)
rfm["F_Score"] = rfm["F_Score"].astype(int)
rfm["M_Score"] = rfm["M_Score"].astype(int)

# Combined RFM Score
rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]
rfm["RFM_Category"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

# 4. Customer Segmentation
def segment_customer(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 4 and m >= 4:
        return "Loyal Customers"
    elif r >= 4 and f <= 2 and m >= 3:
        return "Potential Loyalists"
    elif r >= 4 and f <= 2 and m <= 2:
        return "New Customers"
    elif r >= 3 and f >= 3 and m <= 2:
        return "Promising"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r <= 2 and f >= 2 and m >= 2:
        return "Need Attention"
    elif r <= 2 and f <= 2 and m >= 3:
        return "Hibernating"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

# 5. Segment Analysis
segment_stats = rfm.groupby("Segment").agg(
    Customer_Count=("customer_id", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
    Avg_Satisfaction=("satisfaction_score", "mean")
).round(2)

segment_stats["Revenue_Share_%"] = (segment_stats["Total_Revenue"] / segment_stats["Total_Revenue"].sum() * 100).round(1)
segment_stats = segment_stats.sort_values("Total_Revenue", ascending=False)

print("\n=== RFM Segment Summary ===")
print(segment_stats.to_string())

# 6. Top Customers by Segment
print("\n=== Top 5 Customers per Segment ===")
for segment in rfm["Segment"].unique():
    seg_df = rfm[rfm["Segment"] == segment].nlargest(5, "Monetary")
    print(f"\n{segment} ({len(seg_df)} customers):")
    print(seg_df[["customer_id", "first_name", "Recency", "Frequency", "Monetary",
                  "R_Score", "F_Score", "M_Score", "Segment"]].to_string(index=False))




=== RFM Segment Summary ===
                     Customer_Count  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  Avg_Satisfaction  Revenue_Share_%
Segment                                                                                                                        
At Risk                          13       289.23          22.15       5927.12       77052.62              2.62             34.4
Champions                         7       114.00          28.14       7229.57       50606.98              3.43             22.6
Loyal Customers                   6       209.83          27.50       6974.14       41844.83              2.50             18.7
Lost                             15       216.07           9.93       1371.41       20571.17              3.07              9.2
Need Attention                    4       279.00          11.25       2826.54       11306.16              2.75              5.1
Potential Loyalists               3        94.33          10.00       2761.

In [20]:

SEGMENT_COLORS = {
    "Champions":           "#2E86AB",
    "Loyal Customers":     "#4CB944",
    "Potential Loyalists": "#9C6ADE",
    "New Customers":       "#48C9B0",
    "Promising":           "#F4A261",
    "At Risk":             "#E63946",
    "Need Attention":      "#E9C46A",
    "Hibernating":         "#A0A0A0",
    "Lost":                "#6C6C6C"
}

# Prepare aggregated data
segment_counts = rfm["Segment"].value_counts()
rev_by_seg = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=True)
sat_by_seg = rfm.groupby("Segment")["satisfaction_score"].mean().sort_values(ascending=True)
heatmap_data = rfm.groupby("Segment")[["R_Score", "F_Score", "M_Score"]].mean().round(1)

# Consistent colour order for each chart (fallback to grey if segment not in palette)
def get_segment_colors(seg_list):
    return [SEGMENT_COLORS.get(seg, "#A0A0A0") for seg in seg_list]

# Create 2x3 subplot dashboard
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        "Customer Count by Segment",
        "Total Revenue by Segment",
        "RFM Combined Score Distribution",
        "Recency vs Frequency (colour = Monetary)",
        "Average RFM Scores by Segment",
        "Average Satisfaction by Segment"
    ),
    specs=[
        [{"type": "bar"}, {"type": "bar"}, {"type": "histogram"}],
        [{"type": "scatter"}, {"type": "heatmap"}, {"type": "bar"}]
    ],
    horizontal_spacing=0.12,
    vertical_spacing=0.18
)

# ----- 7.1 Customer Count by Segment -----
fig.add_trace(
    go.Bar(
        y=segment_counts.index.tolist(),
        x=segment_counts.values,
        orientation='h',
        marker_color=get_segment_colors(segment_counts.index),
        text=segment_counts.values,
        textposition='outside',
        name="Count"
    ),
    row=1, col=1
)

# ----- 7.2 Total Revenue by Segment -----
fig.add_trace(
    go.Bar(
        y=rev_by_seg.index.tolist(),
        x=rev_by_seg.values,
        orientation='h',
        marker_color=get_segment_colors(rev_by_seg.index),
        text=[f"${v:,.0f}" for v in rev_by_seg.values],
        textposition='outside',
        name="Revenue"
    ),
    row=1, col=2
)

# ----- 7.3 RFM Score Distribution -----
fig.add_trace(
    go.Histogram(
        x=rfm["RFM_Score"],
        nbinsx=15,
        marker_color="#636EFA",
        opacity=0.85,
        name="RFM Score"
    ),
    row=1, col=3
)

# ----- 7.4 Recency vs Frequency Scatter (colour = Monetary) -----
# Enhance: include segment name in hover text
fig.add_trace(
    go.Scatter(
        x=rfm["Recency"],
        y=rfm["Frequency"],
        mode='markers',
        marker=dict(
            size=9,
            color=rfm["Monetary"],
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="Monetary", x=1.02, len=0.5)
        ),
        customdata=rfm[["customer_id", "Segment"]],
        hovertemplate=(
            "Customer: %{customdata[0]}<br>"
            "Segment: %{customdata[1]}<br>"
            "Recency: %{x} days<br>"
            "Frequency: %{y}<br>"
            "Monetary: %{marker.color:,.2f}<extra></extra>"
        ),
        name="Customers"
    ),
    row=2, col=1
)

# ----- 7.5 Average RFM Scores (Heatmap) -----
fig.add_trace(
    go.Heatmap(
        z=heatmap_data.T.values,            # 3 rows (R,F,M) x n segments
        x=heatmap_data.index,               # segment names
        y=["R_Score", "F_Score", "M_Score"],
        colorscale="RdYlGn",
        zmin=1, zmax=5,
        text=heatmap_data.T.values,
        texttemplate="%{text}",
        colorbar=dict(title="Avg Score", x=1.02, len=0.5)
    ),
    row=2, col=2
)

# ----- 7.6 Average Satisfaction by Segment -----
fig.add_trace(
    go.Bar(
        y=sat_by_seg.index.tolist(),
        x=sat_by_seg.values,
        orientation='h',
        marker_color=get_segment_colors(sat_by_seg.index),
        text=[f"{v:.2f}" for v in sat_by_seg.values],
        textposition='outside',
        name="Satisfaction"
    ),
    row=2, col=3
)

# ----- Layout & Export -----
fig.update_layout(
    title_text="<b>Interactive RFM Analysis Dashboard by <a href='http://github.com/llazdll'>llazdll</a></b>",
    title_font_size=22,
    showlegend=False,
    height=850,
    width=1300,
    hovermode="closest",
    paper_bgcolor="white",
    plot_bgcolor="#F9F9F9"
)

# Axis labels
fig.update_xaxes(title_text="Number of Customers", row=1, col=1)
fig.update_xaxes(title_text="Total Revenue (Currency)", row=1, col=2)
fig.update_xaxes(title_text="RFM Score (3-15)", row=1, col=3)
fig.update_xaxes(title_text="Recency (days)", row=2, col=1)
fig.update_yaxes(title_text="Frequency (purchases)", row=2, col=1)
fig.update_xaxes(title_text="Avg Satisfaction Score", row=2, col=3)

# Show interactive figure (inline for Jupyter, or opens in browser)
fig.show()

# Save as self-contained HTML
fig.write_html("rfm_dashboard.html")
print("✅ Interactive dashboard saved as rfm_dashboard.html")

# (Optional) Static PNG export – requires kaleido: pip install kaleido
# fig.write_image("rfm_dashboard.png", width=1300, height=850, scale=2)

✅ Interactive dashboard saved as rfm_dashboard.html


In [21]:

# 8. Business Questions Answers
print("\n" + "="*60)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("="*60)

# Q1: Who are the champions?
champions = rfm[rfm["Segment"] == "Champions"]
print(f"\n1. CHAMPIONS ({len(champions)} customers):")
print(f"   Revenue share: {segment_stats.loc['Champions', 'Revenue_Share_%']}%")
print(f"   Action: VIP treatment, exclusive offers, referral programs")

# Q2: At Risk customers
at_risk = rfm[rfm["Segment"] == "At Risk"]
print(f"\n2. AT RISK ({len(at_risk)} customers):")
print(f"   High value but inactive recently")
print(f"   Action: Win-back campaigns, personalized outreach, special discounts")

# Q3: Hibernating / Lost
hibernating = rfm[rfm["Segment"] == "Hibernating"]
lost = rfm[rfm["Segment"] == "Lost"]
print(f"\n3. HIBERNATING ({len(hibernating)}) & LOST ({len(lost)}):")
print(f"   Low engagement, low recency")
print(f"   Action: Reactivation campaigns, survey to understand churn reasons")

# Q4: New Customers
new_cust = rfm[rfm["Segment"] == "New Customers"]
print(f"\n4. NEW CUSTOMERS ({len(new_cust)}):")
print(f"   Recent but low frequency/monetary")
print(f"   Action: Onboarding series, second purchase incentives, education")

# Q5: Potential Loyalists
potential = rfm[rfm["Segment"] == "Potential Loyalists"]
print(f"\n5. POTENTIAL LOYALISTS ({len(potential)}):")
print(f"   Recent buyers with good monetary value")
print(f"   Action: Loyalty program enrollment, cross-sell, increase frequency")

# 9. Export Results
output_file = "rfm_analysis_results.xlsx"
with pd.ExcelWriter(output_file) as writer:
    rfm.to_excel(writer, sheet_name="RFM_Scores", index=False)
    segment_stats.to_excel(writer, sheet_name="Segment_Summary")

    # Top customers per segment
    for segment in rfm["Segment"].unique():
        seg_df = rfm[rfm["Segment"] == segment].nlargest(10, "Monetary")
        sheet_name = segment[:31]  # Excel sheet name limit
        seg_df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"\nResults exported to: {output_file}")
print("\nDone!")


BUSINESS INSIGHTS & RECOMMENDATIONS

1. CHAMPIONS (7 customers):
   Revenue share: 22.6%
   Action: VIP treatment, exclusive offers, referral programs

2. AT RISK (13 customers):
   High value but inactive recently
   Action: Win-back campaigns, personalized outreach, special discounts

3. HIBERNATING (0) & LOST (15):
   Low engagement, low recency
   Action: Reactivation campaigns, survey to understand churn reasons

4. NEW CUSTOMERS (6):
   Recent but low frequency/monetary
   Action: Onboarding series, second purchase incentives, education

5. POTENTIAL LOYALISTS (3):
   Recent buyers with good monetary value
   Action: Loyalty program enrollment, cross-sell, increase frequency

Results exported to: rfm_analysis_results.xlsx

Done!
